# Exercise: Catchment delineation, geospatial tools to retrieve site information, automated data collection, and visualization 

Use the [USGS NWIS mapper system](https://apps.usgs.gov/nwismapper/) to identify a NWIS id with high eleveation SNOTEL sites in the Great Salt Lake Basin. Feel free to work backwards and use the [ USDA NRCS SNOTELinteractive map system](https://nwcc-apps.sc.egov.usda.gov/imap/#version=2&elements=&networks=!&states=!&counties=!&hucs=&minElevation=&maxElevation=&elementSelectType=any&activeOnly=true&activeForecastPointsOnly=true&hucLabels=false&hucIdLabels=false&hucParameterLabels=true&stationLabels=&overlays=&hucOverlays=&basinOpacity=75&basinNoDataOpacity=25&basemapOpacity=100&maskOpacity=0&mode=data&openSections=dataElement,parameter,date,basin,options,elements,location,networks&controlsOpen=true&popup=&popupMulti=&popupBasin=&base=esriNgwm&displayType=station&basinType=6&dataElement=WTEQ&depth=-8&parameter=PCTMED&frequency=DAILY&duration=I&customDuration=&dayPart=E&monthPart=E&forecastPubDay=1&forecastExceedance=50&useMixedPast=true&seqColor=1&divColor=7&scaleType=D&scaleMin=&scaleMax=&referencePeriodType=POR&referenceBegin=1991&referenceEnd=2020&minimumYears=20&hucAssociations=true&relativeDate=-1&lat=42.300&lon=-114.300&zoom=4.5) to find a basin upstream of a NWIS site that has 3-4 Snotel sites.

Conduct/repeat the following activities in a new notebook that you create, called: **My_SNOTEL_Analysis.ipynb**:
1. Create a delineated Watershed Map for the area upstream of the NWIS site
2. Retrieve data for the selected sites
3. Process the data to enable plotting and analysis
4. Make a snow report for WY2026
5. Generate a Basin Snow report

*note, make sure you correctly name any files for your particlular watershed of interest


### 1. Create a delineated Watershed Map for the area upstream of the NWIS site

My site = [10129300](https://waterdata.usgs.gov/monitoring-location/10129300/#dataTypeId=continuous-00065-0&period=P7D&showMedian=false) Weber River near Peoa

In [1]:
from pynhd import NLDI, WaterData, NHDPlusHR, GeoConnex
import geopandas as gpd
import pandas as pd
from supporting_scripts import getData, SNOTEL_Analyzer, dataprocessing, mapping
from shapely.geometry import box, Polygon
import os
import datetime
import matplotlib.pyplot as plt
import numpy as np
import warnings
warnings.filterwarnings("ignore")

In [2]:
nldi = NLDI()
usgs_gage_id = "10129300" # NWIS id for my site

In [3]:
#Getting basin geometry
print('Collecting basins...', end='')
basin = nldi.get_basins(usgs_gage_id)
if not os.path.exists('files'):
    os.makedirs('files')
basin.to_file("files/mysitebasin.shp")
print('done')

site_feature = nldi.getfeature_byid("nwissite", f"USGS-{usgs_gage_id}")
upstream_network = nldi.navigate_byid(
    "nwissite", f"USGS-{usgs_gage_id}", "upstreamMain", "flowlines", distance=9999
)

In [4]:
#create map
mapping.basin_mapping(basin, site_feature)

### 2. Retrieve data for the selected sites

In [5]:
# Create geodataframe of all stations
all_stations_gdf = gpd.read_file('https://raw.githubusercontent.com/egagli/snotel_ccss_stations/main/all_stations.geojson').set_index('code')
all_stations_gdf = all_stations_gdf[all_stations_gdf['csvData']==True]

# Use the polygon geometry to select snotel sites that are within the domain
gdf_in_bbox = all_stations_gdf[all_stations_gdf.geometry.within(basin.geometry[0])]

#reset index to have siteid as a column
gdf_in_bbox.reset_index(drop=False, inplace=True)

#make begin and end date a str
gdf_in_bbox['beginDate'] = [datetime.datetime.strftime(gdf_in_bbox['beginDate'][i], "%Y-%m-%d") for i in np.arange(0,len(gdf_in_bbox),1)]
gdf_in_bbox['endDate'] = [datetime.datetime.strftime(gdf_in_bbox['endDate'][i], "%Y-%m-%d") for i in np.arange(0,len(gdf_in_bbox),1)]
gdf_in_bbox

,code,name,network,elevation_m,latitude,longitude,state,HUC,mgrs,mountainRange,beginDate,endDate,csvData,geometry
0,330_UT_SNTL,Beaver Divide,SNOTEL,2523.743896,40.612331,-111.097816,Utah,160202030105,12TVK,Western Rocky Mountains,1978-10-01,2026-03-02,True,POINT (-111.09782 40.61233)
1,763_UT_SNTL,Smith Morehouse,SNOTEL,2325.928711,40.789310,-111.091919,Utah,160201010203,12TVL,Western Rocky Mountains,1978-10-01,2026-03-02,True,POINT (-111.09192 40.78931)


In [6]:
mapping.snotel_mapping(gdf_in_bbox, basin, site_feature)

In [7]:
# Use the getData module to retrieve data 
OutputFolder = 'files/SNOTEL'
if not os.path.exists(OutputFolder):
    os.makedirs(OutputFolder)

for i in gdf_in_bbox.index:
    #getData.getCaliSNOTELData(gdf_in_bbox.name[i], gdf_in_bbox.code[i], gdf_in_bbox.beginDate[i], gdf_in_bbox.endDate[i], OutputFolder)
    getData.getSNOTELData(gdf_in_bbox.name[i], gdf_in_bbox.code[i], gdf_in_bbox.state[i] , gdf_in_bbox.beginDate[i], gdf_in_bbox.endDate[i], OutputFolder)

Start retrieving data for Beaver Divide, 330_UT_SNTL 
 https://wcc.sc.egov.usda.gov/reportGenerator/view_csv/customMultiTimeSeriesGroupByStationReport/daily/start_of_period/330:Utah:SNTL%7Cid=%22%22%7Cname/1978-10-01,2026-03-02/WTEQ::value?fitToScreen=false


KeyError: 1

### 3. Process the data to enable plotting and analysis

### 4. Make a snow report for WY2026

### 5. Generate a Basin Snow report